# Prac 04

For this homework you are going to implement a lane line detector. Lane line detection is crucial for ADAS (Advanced Driver Assistance Systems) systems and, in particular, for LKA (Lane Keep Assist). You will use a [picture](https://en.wikipedia.org/wiki/Lane_departure_warning_system) from a front facing camera (mounted on the car) and will implement the following steps:
* Convert image to gray scale
* Compute edge map
* Apply Hough transform to obtain line parametrizations

In [ ]:
import cv2
import math
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

Let's load and show the camera frame.

In [ ]:
import urllib.request

url = 'https://www.planet-ride.com/wp-content/uploads/2015/02/planet-ride-road-movie.jpg'
urllib.request.urlretrieve(url, "road.jpg")

In [ ]:
img = cv2.imread('road.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, None, fx=0.5, fy=0.5)
plt.imshow(img)

In [ ]:
# Convert image to gray scale
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

plt.imshow(gray, cmap='gray')

In [ ]:
# Obtain edge map
# Hint: you can use Canny edge detector with th_low = 100, th_high = 150
edges = cv2.Canny(img, 100, 150)

# # We are only interseted in the road so we will remove everything above the horizon
edges[0:200] = 0

In [ ]:
h, w = edges.shape

# define the ROI polygon coordinates
roi_vertices = np.array([[[
    (0,  h),
    (328,224),
    (335,224),
    (w, 380) ,
    (w, h)
]]], dtype=np.int32)

mask = np.zeros_like(edges)
cv2.fillPoly(mask, roi_vertices, 255)
masked_edges = cv2.bitwise_and(edges, mask)

In [ ]:
# Let's plot the images
plt.subplot(121), plt.imshow(img), plt.title('Original')
plt.subplot(122), plt.imshow(masked_edges, cmap='gray'), plt.title('Edge map')

In [ ]:
# Apply Hough transform to parametrize the lines
# Hint 1: Offset resolution of 2 pixels and slope resolution of 2 degrees work well in this case
# Hint 2: A suitable value for the accumulator threshold is 190
lines = cv2.HoughLines(masked_edges, 2, 1 * np.pi / 180, 200)
# Let's get rid of the unnecessary dimension
lines = lines[:, 0, :]

In [ ]:
# Plot the resulting Hough lines
result = np.copy(img)

for line in lines:
    rho, theta = line

    a = math.cos(theta)
    b = math.sin(theta)

    x0 = a * rho
    y0 = b * rho

    pt1 = (int(x0 + 1000*(-b)), int(y0 + 1000*(a)))
    pt2 = (int(x0 - 1000*(-b)), int(y0 - 1000*(a)))

    cv2.line(result, pt1, pt2, 255, 1, cv2.LINE_AA)

plt.subplot(121), plt.imshow(masked_edges, cmap='gray'), plt.title('Edge map')
plt.subplot(122), plt.imshow(result, cmap='gray'), plt.title('Hough lines')

The edge map looks good but the Hough lines are too noisy. Let's clean the Hough lines first by removing all lines that we know cannot represent a lane line. In other words, all lines that are approximately horizontal shall be removed. Remember that horizontal lines correspond to theta = 90 degrees.

In [ ]:
# Filter out all lines that are approximately horizontal (+/- 20 degrees).
filtered_lines = []
for line in lines:
    # Extract theta for current line (remember Hough works with radians)
    theta = line[1]
    # Keep line if theta is not horizontal
    if np.abs(theta - np.pi / 2) > np.pi / 9:
        filtered_lines.append(line)

In [ ]:
# Let's plot the resulting filtered lines
result = np.copy(img)

for line in filtered_lines:
    rho, theta = line

    a = math.cos(theta)
    b = math.sin(theta)

    x0 = a * rho
    y0 = b * rho

    pt1 = (int(x0 + 1000*(-b)), int(y0 + 1000*(a)))
    pt2 = (int(x0 - 1000*(-b)), int(y0 - 1000*(a)))

    cv2.line(result, pt1, pt2, 255, 1, cv2.LINE_AA)

plt.subplot(121), plt.imshow(masked_edges, cmap='gray'), plt.title('Edge map')
plt.subplot(122), plt.imshow(result, cmap='gray'), plt.title('Hough lines')

The result is now much better, but still we see some very similar lines. How can we get rid of them?
* Let's apply k-means clustering. It will find the clusters of the 6 we see in the picture lines and use the averages.

In [ ]:
# We will apply k-means clustering to refine the detected lines.
# Don't worry, we will learn about the clustering later in the course :-)
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2).fit(filtered_lines)
kmeans.cluster_centers_

In [ ]:
# Again, let's plot the resulting filtered lines
result = np.copy(img)

for line in kmeans.cluster_centers_:
    rho, theta = line

    a = math.cos(theta)
    b = math.sin(theta)

    x0 = a * rho
    y0 = b * rho

    pt1 = (int(x0 + 1000*(-b)), int(y0 + 1000*(a)))
    pt2 = (int(x0 - 1000*(-b)), int(y0 - 1000*(a)))

    cv2.line(result, pt1, pt2, 255, 1, cv2.LINE_AA)


plt.subplot(121), plt.imshow(masked_edges, cmap='gray'), plt.title('Edge map')
plt.subplot(122), plt.imshow(result, cmap='gray'), plt.title('Hough lines')

### Questions
* Do you see anything strange in the final result?
> No, I see that some techniques produced a really good result for "road" detection

* Do you think the Hough transform resolution is important for obtaining a good result? Why?
 > Yup, since $\rho$ and $\theta$ define the resoultion of our accumulator grid, and a higher resolution (smaller $\rho$ and $\theta$) more accurate results but increases execution time and vice versa. So we need to find a trade-off to prevent a single line from being split into multiple detections or vice versa

* Do you think the Hough transform accumulator threshold is important for obtaining a good result? Why?
> Yup, a higher threshold allows us to find lines more accurately (selecting only those that we are most confident in), but can detect less lines, while a lower threshold can detect more lines, but also introduces noise, because the algo becomes oversensitive. So we need to find a trade-off between detecting all relevant lines and filtering out noise
